# Chapter 10 — Introduction to Artificial Neural Networks with Keras
Companion notebook to `part2_deep_learning/10_ann_with_keras.md`.
Needs `tensorflow` installed. Downloads Fashion MNIST the first time (via Keras).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

print("TensorFlow version:", tf.__version__)

## Load & prep Fashion MNIST

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_valid, X_train = X_train_full[:5000] / 255.0, X_train_full[5000:] / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.0

class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

plt.imshow(X_train[0], cmap="binary")
plt.title(class_names[y_train[0]])
plt.axis("off")
plt.show()

## Sequential API

In [ ]:
model = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(300, activation="relu"),
    keras.layers.Dense(100, activation="relu"),
    keras.layers.Dense(10, activation="softmax"),
])
model.summary()

In [ ]:
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=15,
                     validation_data=(X_valid, y_valid))

In [ ]:
pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True); plt.gca().set_ylim(0, 1)
plt.title("Learning curves")
plt.show()

model.evaluate(X_test, y_test)

In [ ]:
y_proba = model.predict(X_test[:5])
y_pred = y_proba.argmax(axis=-1)
print("Predicted:", [class_names[i] for i in y_pred])
print("Actual:   ", [class_names[i] for i in y_test[:5]])

## Functional API: Wide & Deep on California housing

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_train_full_h, X_test_h, y_train_full_h, y_test_h = train_test_split(housing.data, housing.target, random_state=42)
X_train_h, X_valid_h, y_train_h, y_valid_h = train_test_split(X_train_full_h, y_train_full_h, random_state=42)

scaler = StandardScaler()
X_train_h = scaler.fit_transform(X_train_h)
X_valid_h = scaler.transform(X_valid_h)
X_test_h = scaler.transform(X_test_h)

input_ = keras.layers.Input(shape=X_train_h.shape[1:])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.Concatenate()([input_, hidden2])
output = keras.layers.Dense(1)(concat)
wide_deep_model = keras.Model(inputs=[input_], outputs=[output])

wide_deep_model.compile(loss="mean_squared_error", optimizer="sgd")
history = wide_deep_model.fit(X_train_h, y_train_h, epochs=20,
                               validation_data=(X_valid_h, y_valid_h))
print("Test MSE:", wide_deep_model.evaluate(X_test_h, y_test_h))

## ✍️ Practice
1. Above: is the Fashion MNIST model overfitting, underfitting, or about right? Look at the learning curves.
2. Why can't a network with no/only-linear activations do anything a single linear model couldn't?
3. Add 2 more hidden layers (100 units, relu) to the Fashion MNIST model — does validation accuracy improve? Add EarlyStopping — how many epochs before it stops?
4. Debug: a regression MLP's loss goes `nan` after a few batches — name 2 likely causes.
5. Compare the Wide & Deep model's test MSE above against a plain 2-hidden-layer Sequential model on the same data.

In [ ]:
# Your code / notes here